In [ ]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from langchain_classic.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [7]:
# local embedding model import
MODEL_PATH = Path.home() / "rag-learning-practice" / "models" / "all-MiniLM-L6-v2"

# Local embedding model setup
embeddings = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={
        "device": "cpu",
        "local_files_only": True
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4914.60it/s]


In [ ]:
documents = [
    Document(page_content="LangChain RAG retrieval system documentation."),
    Document(page_content="LangChain retrievers help find relevant documents."),
    Document(page_content="Chroma vector database stores embeddings."),
    Document(page_content="FAISS is a vector similarity search library."),
]


vector_store = Chroma(
    collection_name="ensemble_doc",
    embedding_function=embeddings,
    persist_directory="../all_vector_stores/chroma_ensemble_rectriever"
)

vector_store.add_documents(documents)

['b0b60020-f9db-45af-9b1b-4c2a1fd5b825',
 '287a2c88-90a9-48c9-b070-59412c9df769',
 '97593451-d727-436c-831f-a3b51a916622',
 '14e1cb76-55a3-4126-9be6-141e361ecea6']

In [12]:
vector_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Keyword-based retriever.  BM25 exact keyword match-এ ভালো কাজ করে
bm25_retriever = BM25Retriever.from_documents(
    documents,
    k=3
)

# দুই retriever-কে combine করছি

ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3]   # weights দিয়ে বলছি কোন retriever-এর result কতটা priority পাবে
)

query = "LangChain retrieval documentation"

results = ensemble_retriever.invoke(query)

print("Query:", query)
print("-" * 50)

for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc.page_content}")

Query: LangChain retrieval documentation
--------------------------------------------------
1. LangChain retrievers help find relevant documents.
2. LangChain RAG retrieval system documentation.
3. FAISS is a vector similarity search library.
4. Chroma vector database stores embeddings.
